# TFT Adwer — YOLOv8n Bench + Champion Training

Bu notebook Roboflow `tft-object` dataset'i ile YOLOv8n eğitir ve ONNX'e çevirir.

**Çalıştırma:** Colab'da aç, sırayla cell'leri çalıştır. GPU runtime gerekli (Runtime → Change runtime type → T4 GPU).

**Çıktı:** `bench-yolo.onnx` + `labels.txt` — bunları repo'ya `public/models/` altına koy.

In [ ]:
# 1. GPU kontrol
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
# 2. Ultralytics + Roboflow kur
!pip install ultralytics roboflow onnx onnxruntime --quiet

In [ ]:
# 3. Roboflow dataset indir (tft-object, CC BY 4.0)
# Not: Roboflow API key gerekli. Ücretsiz hesap aç, API key al.
# https://universe.roboflow.com/team-fight-tactics/tft-object
from roboflow import Roboflow
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # ← buraya kendi API key'ini yaz
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("team-fight-tactics").project("tft-object")
version = project.version(1)
dataset = version.download("yolov8")
print(f'Dataset downloaded to: {dataset.location}')

In [ ]:
# 4. data.yaml içeriğini kontrol + düzenle
import os
yaml_path = os.path.join(dataset.location, 'data.yaml')
with open(yaml_path) as f:
    print(f.read())

# Roboflow path'lerini Colab'e göre düzelt
with open(yaml_path) as f:
    content = f.read()
content = content.replace('/content/', '')  # Colab path fix
with open(yaml_path, 'w') as f:
    f.write(content)

In [ ]:
# 5. YOLOv8n eğit (100 epoch, 640 imgsz)
# CPU'da ~6 saat, T4 GPU'da ~30 dk
from ultralytics import YOLO
model = YOLO('yolov8n.pt')  # nano — CPU inference için en hızlı
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,  # GPU
    patience=20,  # early stopping
    save=True,
    project='tft-adwer',
    name='bench-yolo'
)

In [ ]:
# 6. Modeli doğrula
metrics = model.val()
print(f'mAP@50: {metrics.box.map:.3f}')
print(f'mAP@50-95: {metrics.box.map50_95:.3f}')

In [ ]:
# 7. ONNX'e çevir (Next.js'te onnxruntime-node ile kullanmak için)
best_model_path = 'tft-adwer/bench-yolo/weights/best.pt'
model = YOLO(best_model_path)
model.export(format='onnx', imgsz=640, simplify=True, opset=13)
print('ONNX exported: tft-adwer/bench-yolo/weights/best.onnx')

In [ ]:
# 8. labels.txt oluştur (sınıf isimleri)
import yaml
with open(yaml_path) as f:
    data = yaml.safe_load(f)
names = data['names']
if isinstance(names, dict):
    names = [names[k] for k in sorted(names.keys())]
with open('labels.txt', 'w') as f:
    f.write('\n'.join(names))
print(f'labels.txt created: {len(names)} classes')
print(names[:10])  # ilk 10 sınıf

In [ ]:
# 9. Dosyaları indir (Colab'dan locale)
from google.colab import files
files.download('tft-adwer/bench-yolo/weights/best.onnx')
files.download('labels.txt')

## Çıktıları repo'ya koy

İndirilen dosyaları:
- `best.onnx` → `public/models/bench-yolo.onnx`
- `labels.txt` → `public/models/bench-yolo-labels.txt`

Sonra `git push` yap. Next.js otomatik modeli yükleyip kullanacak.